# Water Potability: From-Scratch Models and Missing-Data Strategies

## Project Overview

This notebook implements k-nearest neighbors, Naive Bayes, and decision-tree classifiers from scratch, then evaluates three approaches for handling missing water-quality measurements.

### Technical Coverage

- vectorized Euclidean-distance k-NN
- majority-vote prediction
- Naive Bayes priors and conditional probabilities
- custom binary-feature decision tree
- accuracy-based split selection
- entropy-based split selection
- feature-wise and global binarization variants
- mean imputation
- median imputation
- regression-based imputation
- baseline and tuned classifiers after imputation

## 1. Environment

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import Normalizer
from sklearn.linear_model import LinearRegression


## 2. k-Nearest Neighbors Implemented from Scratch

The custom k-NN implementation computes the full Euclidean-distance matrix with vectorized NumPy operations, sorts neighbors for each test observation, and predicts the most common class among the nearest neighbors.

In [2]:
from scipy import stats

def accuracy(p,y):
  return np.mean(p==y)

def most_common(labels):
  return stats.mode(labels, axis=0, keepdims=False).mode

def distance(x_test,x_train):
  # Returns 2D array dist
  # where dist[i,j] is the Euclidean distance from training example i to test example j
  dist = np.sum(x_train**2,axis=1).reshape(-1,1) # dist = x_train**2
  dist = dist - 2*np.matmul(x_train,x_test.T)    # dist = X_train**2  - 2*X_train*X_test
  dist = dist + np.sum(x_test.T**2,axis=0).reshape(1,-1) # dist = X_train**2  - 2*X_train*X_test + X_test**2 - Not really necessary
  dist = np.sqrt(dist)
  return  dist

def KNN(x_train, y_train, x_test, k):
  d = distance(x_test,x_train)
  neighbors = np.argsort(d,axis=0)[:k]
  pred = most_common(y_train[neighbors])
  return pred


### 20% Test Split

In [6]:
# Load Data
from google.colab import drive
import os

drive.mount('/content/drive')

os.chdir(
    "/content/drive/MyDrive/Machine Learning/7. Comparison of knn, Naive Bayes and Decision trees/lab2_abu/notebooks"
)

print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Machine Learning/7. Comparison of knn, Naive Bayes and Decision trees/lab2_abu/notebooks


In [8]:
import pandas as pd
import numpy as np

# Load Data
water_df = pd.read_excel("../data/water_potability.xlsx")

water_np = water_df.to_numpy()

print("Dataset shape:", water_np.shape)
print(
    "Class counts [not potable, potable]:",
    np.bincount(water_np[:, 9].astype(int))
)
print("Feature matrix shape:", water_np[:, 0:9].shape)

water_df.head()

Dataset shape: (2011, 10)
Class counts [not potable, potable]: [1200  811]
Feature matrix shape: (2011, 9)


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,7.899452,210.734124,15896.365940,6.907203,319.886957,448.666423,18.169921,124.000000,2.853767,1
1,6.145148,197.541072,39657.272110,9.900159,288.157883,319.434033,11.587378,120.030077,4.600886,0
2,5.036454,190.164520,29258.738140,4.991061,300.475925,332.359715,11.055801,116.161622,3.534665,1
3,8.969697,195.744765,9049.682595,7.467068,396.453568,378.528511,17.757697,114.208671,3.983099,0
4,8.285072,151.573778,14402.726700,9.050080,303.081838,322.521815,13.652653,114.034946,4.274661,1


In [9]:


# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])

# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.20, random_state=8684)

# Model Buildings
pred = KNN(x_train, y_train, x_test, 2)


# Model Testing
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier = 0.6551


### 25% Test Split

In [10]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])

# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.25, random_state=7557)

# Model Buildings
pred = KNN(x_train, y_train, x_test, 4)


# Model Testing
print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.5984


### 30% Test Split

In [11]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])

# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.30, random_state=4907)

# Model Buildings
pred = KNN(x_train, y_train, x_test, 2)


# Model Testing
print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.6424


## 3. Naive Bayes Implemented from Scratch

The custom Naive Bayes classifier estimates class priors, class-conditional feature probabilities, and predicts the class with the largest normalized probability product.

In [12]:
np.seterr(divide='ignore', invalid='ignore')

def class_prob(y):
  n_class = len(np.unique(y_train))
  return np.array([np.sum(y == i)/len(y) for i in range(n_class)])

def conditional_prob(x_train,y_train):
  n_class = len(np.unique(y_train))
  return np.array([np.mean(x_train[y_train==i],axis=0) for i in range(n_class)])

def classify(x,pc,pac):
  #print(x.shape)
  p = np.multiply.reduce(np.abs((1 - x) - pac), axis=1) * pc
  p = p/sum(p)
  return np.argmax(p)

def accuracy(p,y):
  return np.mean(p==y)

def Naive_Bayes_Classifier(x_train,y_train,x_test):
  pc = class_prob(y_train)
  pac = conditional_prob(x_train,y_train)
  pred = np.array([classify(x,pc,pac) for x in x_test])
  return pred


### 20% Test Split

In [13]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])

# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.20, random_state=8684)

m = np.mean(x_train)
x_train_binary = np.int32(x_train>m)
x_test_binary = np.int32(x_test>m)

# Model Buildings
pred = Naive_Bayes_Classifier(x_train_binary,y_train,x_test_binary)


# Model Testing
print('Accuracy for Naive Bayes Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for Naive Bayes Classifier= 0.6675


### 25% Test Split

In [14]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])

# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.25, random_state=7557)

m = np.mean(x_train)
x_train_binary = np.int32(x_train>m)
x_test_binary = np.int32(x_test>m)

# Model Buildings
pred = Naive_Bayes_Classifier(x_train_binary,y_train,x_test_binary)


# Model Testing
print('Accuracy for Naive Bayes Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for Naive Bayes Classifier= 0.6302


### 30% Test Split

In [15]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])

# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.30, random_state=4907)

m = np.mean(x_train)
x_train_binary = np.int32(x_train>m)
x_test_binary = np.int32(x_test>m)

# Model Buildings
pred = Naive_Bayes_Classifier(x_train_binary,y_train,x_test_binary)


# Model Testing
print('Accuracy for Naive Bayes Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for Naive Bayes Classifier= 0.6407


## 4. Decision Tree Implemented from Scratch

The custom tree supports two split-selection criteria:

- **classification error:** select the binary feature that minimizes branch misclassification
- **entropy:** select the binary feature that maximizes information gain

The recursive tree builder includes pure-node and no-feature stopping conditions.

In [16]:
def error_count(labels):
  return 0 if len(labels) == 0 else np.minimum(np.sum(labels==0), np.sum(labels==1))

def best_splitting_feature_accuracy(x,y,features):
    best_feature = None
    best_error = 100

    for feature in features:
      left_mistakes = error_count(y[x[:,feature] == 0])
      right_mistakes = error_count(y[x[:,feature] == 1])

      error = float(left_mistakes+right_mistakes)/len(y)

      if error < best_error:
        best_error = error
        best_feature = feature
    return best_feature

def entropy_from_p1(p1):
  p0 = 1-p1
  if p0==0 or p1==0:
    return 0
  return -p0*np.log2(p0) - p1*np.log2(p1)

def best_splitting_feature_entropy(x,y,features):

    best_feature = None
    best_information_gain = -100
    for feature in features:

      a = y[x[:,feature] == 0]
      b = y[x[:,feature] == 1]

      if len(y) == 0:
        information_gain = 0
      elif len(a) ==0 and len(b) == 0:
        information_gain = entropy_from_p1(np.sum(y==0)/len(y))
      elif len(a) == 0:
        information_gain = entropy_from_p1(np.sum(y==0)/len(y)) - (len(b)/len(y))*entropy_from_p1(np.sum(b==0)/len(b))
      elif len(b) == 0:
        information_gain = entropy_from_p1(np.sum(y==0)/len(y)) - (len(a)/len(y))*entropy_from_p1(np.sum(a==0)/len(a))
      else:
        information_gain = entropy_from_p1(np.sum(y==0)/len(y)) - (len(a)/len(y))*entropy_from_p1(np.sum(a==0)/len(a)) - (len(b)/len(y))*entropy_from_p1(np.sum(b==0)/len(b))

      if information_gain > best_information_gain:
        best_information_gain = information_gain
        best_feature = feature
    return best_feature

def create_leaf(y):

  leaf = {'splitting_feature' : None,
          'left' : None,
          'right' : None,
          'is_leaf':  True   }
  leaf['prediction'] = np.argmax(np.bincount(y))
  return leaf

def Decision_Tree_Classifier(x,y,features,criterion,current_depth = 0):
  remaining_features = features[:]
  print("--------------------------------------------------------------------")
  print("Subtree, depth = %s (%s data points)." % (current_depth, len(y)))

  if error_count(y) == 0:
    print("Stopping condition 1 reached.")
    return create_leaf(y)

  if remaining_features == []:
    print("Stopping condition 2 reached.")
    return create_leaf(y)
  if criterion == 'accuracy':
    splitting_feature = best_splitting_feature_accuracy(x,y,features)
  else:
    splitting_feature = best_splitting_feature_entropy(x,y,features)

  left_split = x[:,splitting_feature] == 0
  right_split = x[:,splitting_feature] == 1
  print('splitting_feature ',splitting_feature)
  remaining_features.remove(splitting_feature)
  print("Split on feature %s. (%s, %s)" % (\
                    splitting_feature, np.sum(left_split), np.sum(right_split)))

  if np.sum(left_split) == len(y):
    print("Creating leaf node.")
    return create_leaf(y[left_split])
  if np.sum(right_split) == len(y):
    print("Creating leaf node.")
    return create_leaf(y[right_split])

  left_tree = Decision_Tree_Classifier(x[left_split], y[left_split],remaining_features,criterion, current_depth + 1)
  right_tree = Decision_Tree_Classifier(x[right_split], y[right_split],remaining_features,criterion, current_depth + 1)

  return {'is_leaf'          : False,
          'prediction'       : None,
          'splitting_feature': splitting_feature,
          'left'             : left_tree,
          'right'            : right_tree}

def classify(tree, x):
  if tree['is_leaf']:
    return tree['prediction']
  else:
    split_feature_value = x[tree['splitting_feature']]
    if split_feature_value == 0:
      return classify(tree['left'], x)
    else:
      return classify(tree['right'], x)

### Global-Threshold Binarization

A global training-set mean is used to convert the normalized features into binary attributes before fitting the accuracy-based tree.

In [17]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])
means = np.mean(water_np_normalized,axis=0)

#for i in range(9):
  #water_np_normalized[:,i] = np.int32(water_np_normalized[:,i]>means[i])
# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.20, random_state=8684)

m = np.mean(x_train)
x_train_binary = np.int32(x_train>m)
x_test_binary = np.int32(x_test>m)
#print(m)
#print(np.mean(x_train,axis=0))
#print(np.array([np.unique(x_train[:,i], return_counts=True) for i in range(9)]))
#print(np.array([np.unique(x_test[:,i], return_counts=True) for i in range(9)]))

# Model Buildings
tree = Decision_Tree_Classifier(x_train_binary,y_train,[i for i in range(9)],'accuracy')

pred = np.array([classify(tree, i) for i in x_test_binary])

# Model Testing
print('Accuracy for Decision Tree Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

--------------------------------------------------------------------
Subtree, depth = 0 (1608 data points).
splitting_feature  1
Split on feature 1. (1605, 3)
--------------------------------------------------------------------
Subtree, depth = 1 (1605 data points).
splitting_feature  0
Split on feature 0. (1605, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 1 (3 data points).
Stopping condition 1 reached.
Accuracy for Decision Tree Classifier= 0.6675


### Feature-Wise Binarization with Accuracy Criterion

Each normalized feature is binarized relative to its own feature mean.

#### 20% Test Split

In [18]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])
means = np.mean(water_np_normalized,axis=0)

for i in range(9):
  water_np_normalized[:,i] = np.int32(water_np_normalized[:,i]>means[i])
# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.20, random_state=8684)

#m = np.mean(x_train)
#x_train_binary = np.int32(x_train>m)
#x_test_binary = np.int32(x_test>m)

# Model Buildings
tree = Decision_Tree_Classifier(x_train,y_train,[i for i in range(9)],'accuracy')

#Model Classify
pred = np.array([classify(tree, i) for i in x_test])

# Model Testing
print('Accuracy for Decision Tree Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

--------------------------------------------------------------------
Subtree, depth = 0 (1608 data points).
splitting_feature  0
Split on feature 0. (1038, 570)
--------------------------------------------------------------------
Subtree, depth = 1 (1038 data points).
splitting_feature  1
Split on feature 1. (929, 109)
--------------------------------------------------------------------
Subtree, depth = 2 (929 data points).
splitting_feature  2
Split on feature 2. (0, 929)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 2 (109 data points).
splitting_feature  3
Split on feature 3. (48, 61)
--------------------------------------------------------------------
Subtree, depth = 3 (48 data points).
splitting_feature  4
Split on feature 4. (30, 18)
--------------------------------------------------------------------
Subtree, depth = 4 (30 data points).
splitting_feature  6
Split on feature 6. (20, 10)
---------------------------------

#### 25% Test Split

In [19]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])
means = np.mean(water_np_normalized,axis=0)

for i in range(9):
  water_np_normalized[:,i] = np.int32(water_np_normalized[:,i]>means[i])
# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.25, random_state=7557)

#m = np.mean(x_train)
#x_train_binary = np.int32(x_train>m)
#x_test_binary = np.int32(x_test>m)

# Model Buildings
tree = Decision_Tree_Classifier(x_train,y_train,[i for i in range(9)],'accuracy')

#Model Classify
pred = np.array([classify(tree, i) for i in x_test])

# Model Testing
print('Accuracy for Decision Tree Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

--------------------------------------------------------------------
Subtree, depth = 0 (1508 data points).
splitting_feature  0
Split on feature 0. (974, 534)
--------------------------------------------------------------------
Subtree, depth = 1 (974 data points).
splitting_feature  4
Split on feature 4. (866, 108)
--------------------------------------------------------------------
Subtree, depth = 2 (866 data points).
splitting_feature  1
Split on feature 1. (827, 39)
--------------------------------------------------------------------
Subtree, depth = 3 (827 data points).
splitting_feature  2
Split on feature 2. (0, 827)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 3 (39 data points).
splitting_feature  6
Split on feature 6. (26, 13)
--------------------------------------------------------------------
Subtree, depth = 4 (26 data points).
splitting_feature  2
Split on feature 2. (0, 26)
Creating leaf node.
---------------

#### 30% Test Split

In [20]:
water_df = pd.read_excel('../data/water_potability.xlsx')
water_np = water_df.to_numpy()

y = water_np[:, 9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])
means = np.mean(water_np_normalized, axis=0)

for i in range(9):
    water_np_normalized[:, i] = np.int32(water_np_normalized[:, i] > means[i])

x_train, x_test, y_train, y_test = train_test_split(
    water_np_normalized, y, test_size=0.30, random_state=4907
)

tree = Decision_Tree_Classifier(
    x_train, y_train, [i for i in range(9)], 'accuracy'
)

pred = np.array([classify(tree, i) for i in x_test])
print('Accuracy for Decision Tree Classifier= {:.4f}'.format(
    accuracy_score(pred, y_test)
))

--------------------------------------------------------------------
Subtree, depth = 0 (1407 data points).
splitting_feature  0
Split on feature 0. (921, 486)
--------------------------------------------------------------------
Subtree, depth = 1 (921 data points).
splitting_feature  7
Split on feature 7. (775, 146)
--------------------------------------------------------------------
Subtree, depth = 2 (775 data points).
splitting_feature  1
Split on feature 1. (734, 41)
--------------------------------------------------------------------
Subtree, depth = 3 (734 data points).
splitting_feature  2
Split on feature 2. (0, 734)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 3 (41 data points).
splitting_feature  2
Split on feature 2. (1, 40)
--------------------------------------------------------------------
Subtree, depth = 4 (1 data points).
Stopping condition 1 reached.
--------------------------------------------------------

### Global-Threshold Binarization with Entropy Criterion

#### 20% Test Split

In [22]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])
means = np.mean(water_np_normalized,axis=0)

#for i in range(9):
  #water_np_normalized[:,i] = np.int32(water_np_normalized[:,i]>means[i])
# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.20, random_state=8684)

m = np.mean(x_train)
x_train_binary = np.int32(x_train>m)
x_test_binary = np.int32(x_test>m)
#print(m)
#print(np.mean(x_train,axis=0))
#print(np.array([np.unique(x_train[:,i], return_counts=True) for i in range(9)]))
#print(np.array([np.unique(x_test[:,i], return_counts=True) for i in range(9)]))

# Model Buildings
tree = Decision_Tree_Classifier(x_train_binary,y_train,[i for i in range(9)],'entropy')

pred = np.array([classify(tree, i) for i in x_test_binary])

# Model Testing
print('Accuracy for Decision Tree Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

--------------------------------------------------------------------
Subtree, depth = 0 (1608 data points).
splitting_feature  1
Split on feature 1. (1605, 3)
--------------------------------------------------------------------
Subtree, depth = 1 (1605 data points).
splitting_feature  5
Split on feature 5. (1603, 2)
--------------------------------------------------------------------
Subtree, depth = 2 (1603 data points).
splitting_feature  0
Split on feature 0. (1603, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 2 (2 data points).
splitting_feature  0
Split on feature 0. (2, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 1 (3 data points).
Stopping condition 1 reached.
Accuracy for Decision Tree Classifier= 0.6675


#### 25% Test Split

In [23]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])
means = np.mean(water_np_normalized,axis=0)

#for i in range(9):
  #water_np_normalized[:,i] = np.int32(water_np_normalized[:,i]>means[i])
# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.25, random_state=7557)

m = np.mean(x_train)
x_train_binary = np.int32(x_train>m)
x_test_binary = np.int32(x_test>m)
#print(m)
#print(np.mean(x_train,axis=0))
#print(np.array([np.unique(x_train[:,i], return_counts=True) for i in range(9)]))
#print(np.array([np.unique(x_test[:,i], return_counts=True) for i in range(9)]))

# Model Buildings
tree = Decision_Tree_Classifier(x_train_binary,y_train,[i for i in range(9)],'entropy')

pred = np.array([classify(tree, i) for i in x_test_binary])

# Model Testing
print('Accuracy for Decision Tree Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

--------------------------------------------------------------------
Subtree, depth = 0 (1508 data points).
splitting_feature  1
Split on feature 1. (1504, 4)
--------------------------------------------------------------------
Subtree, depth = 1 (1504 data points).
splitting_feature  4
Split on feature 4. (1503, 1)
--------------------------------------------------------------------
Subtree, depth = 2 (1503 data points).
splitting_feature  5
Split on feature 5. (1502, 1)
--------------------------------------------------------------------
Subtree, depth = 3 (1502 data points).
splitting_feature  0
Split on feature 0. (1502, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 3 (1 data points).
Stopping condition 1 reached.
--------------------------------------------------------------------
Subtree, depth = 2 (1 data points).
Stopping condition 1 reached.
--------------------------------------------------------------------
Subtr

#### 30% Test Split

In [24]:
# Load Data
water_df = pd.read_excel('../data/water_potability.xlsx')

# Convert to Numpy
water_np = water_df.to_numpy()

# Normalization
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])
means = np.mean(water_np_normalized,axis=0)

#for i in range(9):
  #water_np_normalized[:,i] = np.int32(water_np_normalized[:,i]>means[i])
# Split on Training and Test
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.30, random_state=4907)

m = np.mean(x_train)
x_train_binary = np.int32(x_train>m)
x_test_binary = np.int32(x_test>m)
#print(m)
#print(np.mean(x_train,axis=0))
#print(np.array([np.unique(x_train[:,i], return_counts=True) for i in range(9)]))
#print(np.array([np.unique(x_test[:,i], return_counts=True) for i in range(9)]))

# Model Buildings
tree = Decision_Tree_Classifier(x_train_binary,y_train,[i for i in range(9)],'entropy')

pred = np.array([classify(tree, i) for i in x_test_binary])

# Model Testing
print('Accuracy for Decision Tree Classifier= {:.4f}'.format(accuracy_score(pred,y_test)))

--------------------------------------------------------------------
Subtree, depth = 0 (1407 data points).
splitting_feature  7
Split on feature 7. (1406, 1)
--------------------------------------------------------------------
Subtree, depth = 1 (1406 data points).
splitting_feature  1
Split on feature 1. (1404, 2)
--------------------------------------------------------------------
Subtree, depth = 2 (1404 data points).
splitting_feature  4
Split on feature 4. (1403, 1)
--------------------------------------------------------------------
Subtree, depth = 3 (1403 data points).
splitting_feature  5
Split on feature 5. (1401, 2)
--------------------------------------------------------------------
Subtree, depth = 4 (1401 data points).
splitting_feature  0
Split on feature 0. (1401, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 4 (2 data points).
splitting_feature  0
Split on feature 0. (2, 0)
Creating leaf node.
------------

## 5. Dataset with Missing Measurements

The larger dataset contains **3,276 samples**. Missing values occur in three attributes:

- `ph`: 491 missing values
- `Sulfate`: 781 missing values
- `Trihalomethanes`: 162 missing values

In [25]:
water_df_org = pd.read_excel('../data/water_potability_original.xlsx')
water_df_org
#water_np = water_df.to_numpy()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0
...,...,...,...,...,...,...,...,...,...,...
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1
3272,7.808856,193.553212,17329.802160,8.061362,NaN,392.449580,19.903225,NaN,2.798243,1
3273,9.419510,175.762646,33155.578218,7.350233,NaN,432.044783,11.039070,69.845400,3.298875,1
3274,5.126763,230.603758,11983.869376,6.303357,NaN,402.883113,11.168946,77.488213,4.708658,1


In [26]:
print(water_df_org.shape)
print(water_df_org.isnull().sum())

(3276, 10)
ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64


## 6. Mean Imputation

Missing values are replaced with mean estimates before evaluating both default and tuned classifiers.

In [27]:
train, test = train_test_split(water_df_org, test_size=0.20, random_state=8684)


test['ph'] = test['ph'].replace(np.nan, train['ph'].mean())
test['Sulfate'] = test['Sulfate'].replace(np.nan, test['Sulfate'].mean())
test['Trihalomethanes'] = test['Trihalomethanes'].replace(np.nan, test['Trihalomethanes'].mean())

train['ph'] = train['ph'].replace(np.nan, train['ph'].mean())
train['Sulfate'] = train['Sulfate'].replace(np.nan, train['Sulfate'].mean())
train['Trihalomethanes'] = train['Trihalomethanes'].replace(np.nan, train['Trihalomethanes'].mean())

print(train.isnull().sum())

print(test.isnull().sum())

train_np = train.to_numpy()
test_np = test.to_numpy()

x_train = train_np[:,0:9]
y_train = train_np[:,9]
x_test = test_np[:,0:9]
y_test = test_np[:,9]

print(x_train.shape)
print(y_train.shape)

ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64
ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64
(2620, 9)
(2620,)


### Baseline Models

In [28]:
model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for MultinomialNB= {:.4f}'.format(accuracy_score(pred,y_test)))


model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for BernoulliNB= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for DecisionTreeClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.5549
Accuracy for MultinomialNB= 0.5061
Accuracy for BernoulliNB= 0.5976
Accuracy for DecisionTreeClassifier= 0.5823


### Tuned Models

In [29]:
model =  KNeighborsClassifier(n_neighbors=2, weights='uniform', algorithm='ball_tree', p=2)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB(alpha=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for MultinomialNB= {:.4f}'.format(accuracy_score(pred,y_test)))


model =  BernoulliNB(alpha=1.0, binarize=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for BernoulliNB= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=11, min_samples_split=3, max_features='sqrt')
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for DecisionTreeClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.5808
Accuracy for MultinomialNB= 0.5061
Accuracy for BernoulliNB= 0.5976
Accuracy for DecisionTreeClassifier= 0.5854


## 7. Median Imputation

The same evaluation is repeated after replacing missing values with median estimates.

In [30]:
train, test = train_test_split(water_df_org, test_size=0.20, random_state=8684)


test['ph'] = test['ph'].replace(np.nan, train['ph'].median())
test['Sulfate'] = test['Sulfate'].replace(np.nan, test['Sulfate'].median())
test['Trihalomethanes'] = test['Trihalomethanes'].replace(np.nan, test['Trihalomethanes'].median())

train['ph'] = train['ph'].replace(np.nan, train['ph'].median())
train['Sulfate'] = train['Sulfate'].replace(np.nan, train['Sulfate'].median())
train['Trihalomethanes'] = train['Trihalomethanes'].replace(np.nan, train['Trihalomethanes'].median())

print(train.isnull().sum())

print(test.isnull().sum())

train_np = train.to_numpy()
test_np = test.to_numpy()

x_train = train_np[:,0:9]
y_train = train_np[:,9]
x_test = test_np[:,0:9]
y_test = test_np[:,9]

print(x_train.shape)
print(y_train.shape)

ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64
ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64
(2620, 9)
(2620,)


### Baseline Models

In [31]:
model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for MultinomialNB= {:.4f}'.format(accuracy_score(pred,y_test)))


model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for BernoulliNB= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for DecisionTreeClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.5549
Accuracy for MultinomialNB= 0.5046
Accuracy for BernoulliNB= 0.5976
Accuracy for DecisionTreeClassifier= 0.5808


### Tuned Models

In [32]:
model =  KNeighborsClassifier(n_neighbors=2, weights='uniform', algorithm='ball_tree', p=2)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB(alpha=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for MultinomialNB= {:.4f}'.format(accuracy_score(pred,y_test)))


model =  BernoulliNB(alpha=1.0, binarize=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for BernoulliNB= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=11, min_samples_split=3, max_features='sqrt')
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for DecisionTreeClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.5808
Accuracy for MultinomialNB= 0.5046
Accuracy for BernoulliNB= 0.5976
Accuracy for DecisionTreeClassifier= 0.6280


## 8. Regression-Based Imputation

Linear regression models estimate the missing values for `ph`, `Sulfate`, and `Trihalomethanes` sequentially. The completed dataset is then evaluated with the same baseline and tuned classifiers.

### Predicting Missing pH

In [33]:
water_df_org = pd.read_excel('../data/water_potability_original.xlsx')
water_df_org

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0
...,...,...,...,...,...,...,...,...,...,...
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1
3272,7.808856,193.553212,17329.802160,8.061362,NaN,392.449580,19.903225,NaN,2.798243,1
3273,9.419510,175.762646,33155.578218,7.350233,NaN,432.044783,11.039070,69.845400,3.298875,1
3274,5.126763,230.603758,11983.869376,6.303357,NaN,402.883113,11.168946,77.488213,4.708658,1


In [ ]:
water_df_org.columns

In [34]:
data = water_df_org.copy()
data = data[['ph', 'Hardness', 'Solids', 'Chloramines', 'Conductivity', 'Organic_carbon', 'Turbidity', 'Potability']]

isnull = data.ph.isnull()

test_data = data[data["ph"].isnull()]

data.dropna(inplace=True)

y_train = data["ph"]
X_train = data.drop("ph", axis=1)
X_test = test_data.drop("ph", axis=1)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

water_df_org.loc[isnull, 'ph'] = y_pred
print(water_df_org.isnull().sum())

ph                   0
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64


### Predicting Missing Sulfate

In [35]:
data = water_df_org.copy()
data = data[['Sulfate', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Conductivity', 'Organic_carbon', 'Turbidity', 'Potability']]

isnull = data.Sulfate.isnull()

test_data = data[data["Sulfate"].isnull()]

data.dropna(inplace=True)

y_train = data["Sulfate"]
X_train = data.drop("Sulfate", axis=1)
X_test = test_data.drop("Sulfate", axis=1)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

water_df_org.loc[isnull, 'Sulfate'] = y_pred
print(water_df_org.isnull().sum())

ph                   0
Hardness             0
Solids               0
Chloramines          0
Sulfate              0
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64


### Predicting Missing Trihalomethanes

In [36]:
data = water_df_org.copy()
data = data[['Trihalomethanes','Sulfate', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Conductivity', 'Organic_carbon', 'Turbidity', 'Potability']]

isnull = data.Trihalomethanes.isnull()

test_data = data[data["Trihalomethanes"].isnull()]

data.dropna(inplace=True)

y_train = data["Trihalomethanes"]
X_train = data.drop("Trihalomethanes", axis=1)
X_test = test_data.drop("Trihalomethanes", axis=1)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

water_df_org.loc[isnull, 'Trihalomethanes'] = y_pred
print(water_df_org.isnull().sum())

ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64


### Completed Dataset Split

In [37]:
train, test = train_test_split(water_df_org, test_size=0.20, random_state=8684)


train_np = train.to_numpy()
test_np = test.to_numpy()

x_train = train_np[:,0:9]
y_train = train_np[:,9]
x_test = test_np[:,0:9]
y_test = test_np[:,9]

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(2620, 9)
(2620,)
(656, 9)
(656,)


### Baseline Models

In [38]:
model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for MultinomialNB= {:.4f}'.format(accuracy_score(pred,y_test)))


model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for BernoulliNB= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for DecisionTreeClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.5534
Accuracy for MultinomialNB= 0.5091
Accuracy for BernoulliNB= 0.5976
Accuracy for DecisionTreeClassifier= 0.5960


### Tuned Models

In [39]:
model =  KNeighborsClassifier(n_neighbors=2, weights='uniform', algorithm='ball_tree', p=2)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for KNeighborsClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB(alpha=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for MultinomialNB= {:.4f}'.format(accuracy_score(pred,y_test)))


model =  BernoulliNB(alpha=1.0, binarize=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for BernoulliNB= {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=11, min_samples_split=3, max_features='sqrt')
model.fit(x_train, y_train)
pred = model.predict(x_test)

print('Accuracy for DecisionTreeClassifier= {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier= 0.5747
Accuracy for MultinomialNB= 0.5091
Accuracy for BernoulliNB= 0.5976
Accuracy for DecisionTreeClassifier= 0.6021


## 9. Comparative Results

### From-Scratch Classifiers

| Model | 20% Test | 25% Test | 30% Test |
|---|---:|---:|---:|
| k-NN | 0.6551 | 0.5984 | 0.6424 |
| Naive Bayes | 0.6675 | 0.6302 | 0.6407 |
| Decision tree — accuracy criterion | **0.6774** | 0.6123 | 0.6308 |
| Decision tree — entropy criterion | 0.6675 | **0.6302** | **0.6407** |

The additional global-threshold accuracy-tree experiment on the 20% split produces **0.6675** accuracy.

### Missing-Value Strategies — Baseline Models

| Model | Mean | Median | Regression-Based |
|---|---:|---:|---:|
| KNeighborsClassifier | **0.5549** | **0.5549** | 0.5534 |
| MultinomialNB | 0.5061 | 0.5046 | **0.5091** |
| BernoulliNB | 0.5976 | 0.5976 | 0.5976 |
| DecisionTreeClassifier | 0.5915 | 0.5838 | **0.5945** |

### Missing-Value Strategies — Tuned Models

| Model | Mean | Median | Regression-Based |
|---|---:|---:|---:|
| KNeighborsClassifier | **0.5808** | **0.5808** | 0.5747 |
| MultinomialNB | 0.5061 | 0.5046 | **0.5091** |
| BernoulliNB | 0.5976 | 0.5976 | 0.5976 |
| DecisionTreeClassifier | 0.6021 | 0.5899 | **0.6265** |

## 10. Key Findings

- The custom implementations achieve test accuracies in the same general range as the library-based models.
- The accuracy-based custom decision tree reaches the strongest saved from-scratch result, 0.6774 on the 20% test split.
- BernoulliNB is insensitive to the three missing-value strategies in the saved evaluation.
- Regression-based imputation provides the strongest tuned decision-tree result, 0.6265.
- Mean and median imputation produce identical tuned k-NN accuracy in the saved experiments.